# LOF - multi-run experiments

In [2]:
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample
from optuna_utils import run_study
from metrics import find_best_f1_threshold, minmax_scale_scores, evaluate_scores, print_metrics
from results import build_experiment_record, save_record_json, get_memory_mb

In [3]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200

DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
MODEL_TYPE = "LOF"
FUSION_STRATEGY = "none"

RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000,  n_val_opt=3000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run3_large_opt_sample"),
]

In [4]:
def make_objective(train_x, train_y, val_x, val_y):
    def objective(trial):
        n_pos = int(val_y.sum())
        n_neg = len(val_y) - n_pos
        if n_pos < 5 or n_neg < 5:
            raise optuna.exceptions.TrialPruned()

        n_neighbors = trial.suggest_int("n_neighbors", 5, 100, log=True)
        leaf_size = trial.suggest_int("leaf_size", 10, 60, step=10)
        metric = trial.suggest_categorical("metric", ["minkowski", "euclidean", "manhattan"])
        contamination = trial.suggest_float("contamination", 0.01, 0.5)
        p = trial.suggest_int("p", 1, 2) if metric == "minkowski" else 2

        try:
            clf = LocalOutlierFactor(
                n_neighbors=n_neighbors,
                leaf_size=leaf_size,
                metric=metric,
                p=p,
                contamination=contamination,
                novelty=True,
                n_jobs=-1,
            )
            clf.fit(train_x)
            val_scores = -clf.score_samples(val_x)
            auc = roc_auc_score(val_y, val_scores)
        except ValueError:
            raise optuna.exceptions.TrialPruned()

        return auc
    return objective

In [5]:
def fit_and_score_lof(params, train_x, train_y, val_x, test_x):
    clf = LocalOutlierFactor(
        n_neighbors=params["n_neighbors"],
        leaf_size=params["leaf_size"],
        metric=params["metric"],
        p=params.get("p", 2),
        contamination=params["contamination"],
        novelty=True,
        n_jobs=-1,
    )

    gc.collect()
    mem_before = get_memory_mb()

    start_train = time.time()
    clf.fit(train_x)
    runtime_train = time.time() - start_train
    mem_after_train = get_memory_mb()

    val_scores_raw = -clf.score_samples(val_x)
    scores_val = minmax_scale_scores(val_scores_raw)

    start_inference = time.time()
    test_scores_raw = -clf.score_samples(test_x)
    runtime_inference = time.time() - start_inference
    mem_after_inference = get_memory_mb()

    scores_test = minmax_scale_scores(test_scores_raw)
    memory_peak = max(mem_before, mem_after_train, mem_after_inference)

    return clf, scores_val, scores_test, runtime_train, runtime_inference, memory_peak

In [6]:
def run_experiment(dataset, run_cfg):
    run_index = run_cfg["run_index"]
    study_name = f"LOF_{dataset}_run{run_index}"

    train_x, train_y, val_x, val_y = make_optuna_subsample(
        dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"]
    )
    objective = make_objective(train_x, train_y, val_x, val_y)
    study = run_study(objective, study_name, SEED, N_TRIALS, results_dir=RESULTS_DIR)

    train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(
        dataset, SEED,
        run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"],
    )

    clf, scores_val, scores_test, runtime_train, runtime_inference, memory_peak = fit_and_score_lof(
        study.best_params, train_x, train_y, val_x, test_x
    )

    best_threshold, best_f1_val, best_prec_val, best_rec_val = find_best_f1_threshold(val_y, scores_val)
    print(f"[{dataset} run{run_index}] threshold={best_threshold:.4f} "
          f"F1={best_f1_val:.4f} P={best_prec_val:.4f} R={best_rec_val:.4f}")

    metrics = evaluate_scores(test_y, scores_test, threshold=best_threshold)
    print_metrics(f"LOF final - {dataset} run{run_index}", metrics)

    model_dir = MODELS_DIR / MODEL_TYPE
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / f"{dataset}_run{run_index}.joblib"
    joblib.dump(clf, model_path)

    record = build_experiment_record(
        dataset_name=dataset,
        dataset_version=DATASET_VERSION,
        split_method=SPLIT_METHOD,
        seed=SEED,
        preprocessing_version=PREPROCESSING_VERSION,
        model_type=MODEL_TYPE,
        fusion_strategy=FUSION_STRATEGY,
        hyperparameters=study.best_params,
        threshold=best_threshold,
        scores_test=scores_test,
        test_y=test_y,
        runtime_train=runtime_train,
        runtime_inference=runtime_inference,
        memory_peak=memory_peak,
        notes=run_cfg["notes"],
    )
    save_record_json(record, RESULTS_DIR, run_index, MODEL_TYPE, dataset)
    return record

In [7]:
all_records = []

In [7]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

[I 2026-07-27 10:56:40,826] A new study created in memory with name: LOF_CICIDS_run1


CICIDS optuna subsample -> train: 7000, val: 3000 (val positives: 726)


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-27 10:56:43,145] Trial 0 finished with value: 0.6952612597551431 and parameters: {'n_neighbors': 66, 'leaf_size': 20, 'metric': 'euclidean', 'contamination': 0.2757254682642995}. Best is trial 0 with value: 0.6952612597551431.
[I 2026-07-27 10:56:43,964] Trial 1 finished with value: 0.800300316671149 and parameters: {'n_neighbors': 43, 'leaf_size': 60, 'metric': 'minkowski', 'contamination': 0.362951259351822, 'p': 1}. Best is trial 1 with value: 0.800300316671149.
[I 2026-07-27 10:56:44,770] Trial 2 finished with value: 0.6087445575932022 and parameters: {'n_neighbors': 16, 'leaf_size': 40, 'metric': 'manhattan', 'contamination': 0.4220506703403939}. Best is trial 1 with value: 0.800300316671149.
[I 2026-07-27 10:56:45,583] Trial 3 finished with value: 0.753573756272245 and parameters: {'n_neighbors': 56, 'leaf_size': 10, 'metric': 'manhattan', 'contamination': 0.24751720505560543}. Best is trial 1 with value: 0.800300316671149.
[I 2026-07-27 10:56:45,964] Trial 4 finished 

,experiment_id,dataset_name,dataset_version,split_method,seed,preprocessing_version,model_type,fusion_strategy,hyperparameters,threshold,...,AUC_ROC,AUC_PR,Precision,Recall,F1,ConfusionMatrix,runtime_train,runtime_inference,memory_peak,notes
0,c278dda5-cfb1-4d09-80ec-b3cd40a206b6,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 42, 'leaf_size': 40, 'metric':...",7.906182e-11,...,0.798456,0.492324,0.483001,0.826058,0.609578,"[[54374, 21411], [4212, 20003]]",220.067883,143.144763,2089.921875,run1_small_opt_sample


In [8]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

[I 2026-07-27 11:09:16,314] A new study created in memory with name: LOF_UNSW_NB15_run1


UNSW_NB15 optuna subsample -> train: 7000, val: 3000 (val positives: 2042)


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-27 11:09:16,765] Trial 0 finished with value: 0.8518118468323862 and parameters: {'n_neighbors': 66, 'leaf_size': 20, 'metric': 'euclidean', 'contamination': 0.2757254682642995}. Best is trial 0 with value: 0.8518118468323862.
[I 2026-07-27 11:09:18,355] Trial 1 finished with value: 0.8687550990780254 and parameters: {'n_neighbors': 43, 'leaf_size': 60, 'metric': 'minkowski', 'contamination': 0.362951259351822, 'p': 1}. Best is trial 1 with value: 0.8687550990780254.
[I 2026-07-27 11:09:19,918] Trial 2 finished with value: 0.8872659024780242 and parameters: {'n_neighbors': 16, 'leaf_size': 40, 'metric': 'manhattan', 'contamination': 0.4220506703403939}. Best is trial 2 with value: 0.8872659024780242.
[I 2026-07-27 11:09:21,494] Trial 3 finished with value: 0.8668519544676614 and parameters: {'n_neighbors': 56, 'leaf_size': 10, 'metric': 'manhattan', 'contamination': 0.24751720505560543}. Best is trial 2 with value: 0.8872659024780242.
[I 2026-07-27 11:09:21,925] Trial 4 fini

,experiment_id,dataset_name,dataset_version,split_method,seed,preprocessing_version,model_type,fusion_strategy,hyperparameters,threshold,...,AUC_ROC,AUC_PR,Precision,Recall,F1,ConfusionMatrix,runtime_train,runtime_inference,memory_peak,notes
0,c278dda5-cfb1-4d09-80ec-b3cd40a206b6,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 42, 'leaf_size': 40, 'metric':...",7.906182e-11,...,0.798456,0.492324,0.483001,0.826058,0.609578,"[[54374, 21411], [4212, 20003]]",220.067883,143.144763,2089.921875,run1_small_opt_sample
1,91a03e98-3e85-4d4a-9618-e133b250c041,UNSW_NB15,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 14, 'leaf_size': 40, 'metric':...",5.167825e-12,...,0.896154,0.927284,0.773328,0.872033,0.819720,"[[25413, 11587], [5801, 39531]]",27.608088,59.974188,2393.875000,run1_small_opt_sample


In [9]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

[I 2026-07-27 11:16:19,410] A new study created in memory with name: LOF_CICIDS_run2


CICIDS optuna subsample -> train: 21000, val: 9000 (val positives: 2177)


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-27 11:16:22,609] Trial 0 finished with value: 0.6710890190041237 and parameters: {'n_neighbors': 66, 'leaf_size': 20, 'metric': 'euclidean', 'contamination': 0.2757254682642995}. Best is trial 0 with value: 0.6710890190041237.
[I 2026-07-27 11:16:28,630] Trial 1 finished with value: 0.5971803872591497 and parameters: {'n_neighbors': 43, 'leaf_size': 60, 'metric': 'minkowski', 'contamination': 0.362951259351822, 'p': 1}. Best is trial 0 with value: 0.6710890190041237.
[I 2026-07-27 11:16:34,661] Trial 2 finished with value: 0.539928277662808 and parameters: {'n_neighbors': 16, 'leaf_size': 40, 'metric': 'manhattan', 'contamination': 0.4220506703403939}. Best is trial 0 with value: 0.6710890190041237.
[I 2026-07-27 11:16:40,533] Trial 3 finished with value: 0.6565762430041705 and parameters: {'n_neighbors': 56, 'leaf_size': 10, 'metric': 'manhattan', 'contamination': 0.24751720505560543}. Best is trial 0 with value: 0.6710890190041237.
[I 2026-07-27 11:16:43,581] Trial 4 finis

,experiment_id,dataset_name,dataset_version,split_method,seed,preprocessing_version,model_type,fusion_strategy,hyperparameters,threshold,...,AUC_ROC,AUC_PR,Precision,Recall,F1,ConfusionMatrix,runtime_train,runtime_inference,memory_peak,notes
0,c278dda5-cfb1-4d09-80ec-b3cd40a206b6,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 42, 'leaf_size': 40, 'metric':...",7.906182e-11,...,0.798456,0.492324,0.483001,0.826058,0.609578,"[[54374, 21411], [4212, 20003]]",220.067883,143.144763,2089.921875,run1_small_opt_sample
1,91a03e98-3e85-4d4a-9618-e133b250c041,UNSW_NB15,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 14, 'leaf_size': 40, 'metric':...",5.167825e-12,...,0.896154,0.927284,0.773328,0.872033,0.819720,"[[25413, 11587], [5801, 39531]]",27.608088,59.974188,2393.875000,run1_small_opt_sample
2,8ca423e4-6f7e-4193-b04f-7212104a9116,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 100, 'leaf_size': 20, 'metric'...",6.054940e-11,...,0.811459,0.440691,0.482514,0.873467,0.621631,"[[53101, 22684], [3064, 21151]]",166.370851,109.645365,2481.906250,run2_medium_opt_sample


In [10]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

[I 2026-07-27 11:39:31,718] A new study created in memory with name: LOF_UNSW_NB15_run2


UNSW_NB15 optuna subsample -> train: 21000, val: 9000 (val positives: 6126)


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-27 11:39:35,311] Trial 0 finished with value: 0.8777401261061208 and parameters: {'n_neighbors': 66, 'leaf_size': 20, 'metric': 'euclidean', 'contamination': 0.2757254682642995}. Best is trial 0 with value: 0.8777401261061208.
[I 2026-07-27 11:39:47,798] Trial 1 finished with value: 0.9006278440388129 and parameters: {'n_neighbors': 43, 'leaf_size': 60, 'metric': 'minkowski', 'contamination': 0.362951259351822, 'p': 1}. Best is trial 1 with value: 0.9006278440388129.
[I 2026-07-27 11:40:00,072] Trial 2 finished with value: 0.8758590192821543 and parameters: {'n_neighbors': 16, 'leaf_size': 40, 'metric': 'manhattan', 'contamination': 0.4220506703403939}. Best is trial 1 with value: 0.9006278440388129.
[I 2026-07-27 11:40:12,482] Trial 3 finished with value: 0.8969573030384201 and parameters: {'n_neighbors': 56, 'leaf_size': 10, 'metric': 'manhattan', 'contamination': 0.24751720505560543}. Best is trial 1 with value: 0.9006278440388129.
[I 2026-07-27 11:40:15,977] Trial 4 fini

,experiment_id,dataset_name,dataset_version,split_method,seed,preprocessing_version,model_type,fusion_strategy,hyperparameters,threshold,...,AUC_ROC,AUC_PR,Precision,Recall,F1,ConfusionMatrix,runtime_train,runtime_inference,memory_peak,notes
0,c278dda5-cfb1-4d09-80ec-b3cd40a206b6,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 42, 'leaf_size': 40, 'metric':...",7.906182e-11,...,0.798456,0.492324,0.483001,0.826058,0.609578,"[[54374, 21411], [4212, 20003]]",220.067883,143.144763,2089.921875,run1_small_opt_sample
1,91a03e98-3e85-4d4a-9618-e133b250c041,UNSW_NB15,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 14, 'leaf_size': 40, 'metric':...",5.167825e-12,...,0.896154,0.927284,0.773328,0.872033,0.819720,"[[25413, 11587], [5801, 39531]]",27.608088,59.974188,2393.875000,run1_small_opt_sample
2,8ca423e4-6f7e-4193-b04f-7212104a9116,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 100, 'leaf_size': 20, 'metric'...",6.054940e-11,...,0.811459,0.440691,0.482514,0.873467,0.621631,"[[53101, 22684], [3064, 21151]]",166.370851,109.645365,2481.906250,run2_medium_opt_sample
3,b129d525-e0f6-4aa7-8ae6-d0b0e0f871af,UNSW_NB15,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 48, 'leaf_size': 30, 'metric':...",7.278875e-12,...,0.896950,0.918083,0.730715,0.906446,0.809149,"[[21857, 15143], [4241, 41091]]",29.233860,60.652467,2425.179688,run2_medium_opt_sample


In [11]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)

[I 2026-07-27 12:19:40,056] A new study created in memory with name: LOF_CICIDS_run3


CICIDS optuna subsample -> train: 35000, val: 15000 (val positives: 3629)


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-27 12:19:48,174] Trial 0 finished with value: 0.5917905015681555 and parameters: {'n_neighbors': 66, 'leaf_size': 20, 'metric': 'euclidean', 'contamination': 0.2757254682642995}. Best is trial 0 with value: 0.5917905015681555.
[I 2026-07-27 12:20:04,125] Trial 1 finished with value: 0.5259932307871112 and parameters: {'n_neighbors': 43, 'leaf_size': 60, 'metric': 'minkowski', 'contamination': 0.362951259351822, 'p': 1}. Best is trial 0 with value: 0.5917905015681555.
[I 2026-07-27 12:20:21,209] Trial 2 finished with value: 0.5540896178802176 and parameters: {'n_neighbors': 16, 'leaf_size': 40, 'metric': 'manhattan', 'contamination': 0.4220506703403939}. Best is trial 0 with value: 0.5917905015681555.
[I 2026-07-27 12:20:37,343] Trial 3 finished with value: 0.5627919534154544 and parameters: {'n_neighbors': 56, 'leaf_size': 10, 'metric': 'manhattan', 'contamination': 0.24751720505560543}. Best is trial 0 with value: 0.5917905015681555.
[I 2026-07-27 12:20:45,223] Trial 4 fini

,experiment_id,dataset_name,dataset_version,split_method,seed,preprocessing_version,model_type,fusion_strategy,hyperparameters,threshold,...,AUC_ROC,AUC_PR,Precision,Recall,F1,ConfusionMatrix,runtime_train,runtime_inference,memory_peak,notes
0,c278dda5-cfb1-4d09-80ec-b3cd40a206b6,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 42, 'leaf_size': 40, 'metric':...",7.906182e-11,...,0.798456,0.492324,0.483001,0.826058,0.609578,"[[54374, 21411], [4212, 20003]]",220.067883,143.144763,2089.921875,run1_small_opt_sample
1,91a03e98-3e85-4d4a-9618-e133b250c041,UNSW_NB15,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 14, 'leaf_size': 40, 'metric':...",5.167825e-12,...,0.896154,0.927284,0.773328,0.872033,0.819720,"[[25413, 11587], [5801, 39531]]",27.608088,59.974188,2393.875000,run1_small_opt_sample
2,8ca423e4-6f7e-4193-b04f-7212104a9116,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 100, 'leaf_size': 20, 'metric'...",6.054940e-11,...,0.811459,0.440691,0.482514,0.873467,0.621631,"[[53101, 22684], [3064, 21151]]",166.370851,109.645365,2481.906250,run2_medium_opt_sample
3,b129d525-e0f6-4aa7-8ae6-d0b0e0f871af,UNSW_NB15,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 48, 'leaf_size': 30, 'metric':...",7.278875e-12,...,0.896950,0.918083,0.730715,0.906446,0.809149,"[[21857, 15143], [4241, 41091]]",29.233860,60.652467,2425.179688,run2_medium_opt_sample
4,dfcddaeb-79fd-4681-9ae0-17096d4ab0eb,CICIDS,v1,stratified_train_val_test_fixed_seed,29,v1,LOF,none,"{'n_neighbors': 13, 'leaf_size': 30, 'metric':...",8.137649e-10,...,0.853417,0.578099,0.631658,0.572001,0.600351,"[[67708, 8077], [10364, 13851]]",17.667551,11.611569,2458.714844,run3_large_opt_sample


In [8]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)

[I 2026-07-27 14:29:14,168] A new study created in memory with name: LOF_UNSW_NB15_run3


UNSW_NB15 optuna subsample -> train: 35000, val: 15000 (val positives: 10209)


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-07-27 14:29:25,551] Trial 0 finished with value: 0.8760852472614774 and parameters: {'n_neighbors': 66, 'leaf_size': 20, 'metric': 'euclidean', 'contamination': 0.2757254682642995}. Best is trial 0 with value: 0.8760852472614774.
[I 2026-07-27 14:30:06,098] Trial 1 finished with value: 0.8882429954506033 and parameters: {'n_neighbors': 43, 'leaf_size': 60, 'metric': 'minkowski', 'contamination': 0.362951259351822, 'p': 1}. Best is trial 1 with value: 0.8882429954506033.
[I 2026-07-27 14:30:49,478] Trial 2 finished with value: 0.8713268803893839 and parameters: {'n_neighbors': 16, 'leaf_size': 40, 'metric': 'manhattan', 'contamination': 0.4220506703403939}. Best is trial 1 with value: 0.8882429954506033.
[I 2026-07-27 14:31:31,972] Trial 3 finished with value: 0.8948630990711987 and parameters: {'n_neighbors': 56, 'leaf_size': 10, 'metric': 'manhattan', 'contamination': 0.24751720505560543}. Best is trial 3 with value: 0.8948630990711987.
[I 2026-07-27 14:31:42,538] Trial 4 fini

KeyboardInterrupt: 